##  Centroid-Based Cluster Explainability

This block implements **geometric explainability** for the HDBSCAN clusters using document embeddings. Instead of relying on density scores or keyword extraction, we explain clusters using distances in embedding space.

First, we load the cluster assignments and join them with the original document embeddings and text. This ensures each document has its cluster label, embedding vector, and content available for analysis.

Next, we normalize all embeddings so that the dot product between vectors becomes cosine similarity. This allows us to measure how close a document is to its cluster center using a simple and efficient similarity calculation.

For each cluster, we compute a centroid by averaging all embeddings in that cluster and normalizing the result. The centroid represents the geometric center of the cluster in embedding space.

We then compute the similarity of each document to its own centroid and to all other cluster centroids. This allows us to calculate a **centroid margin**, which measures how clearly a document belongs to its assigned cluster compared to competing clusters.

Using these scores, we identify two important types of documents. The first are **representative (prototype) documents**, which are the most central examples of each cluster. The second are **boundary documents**, which have low margin scores and may sit between clusters.

Finally, we save these explainability metrics to a table so they can be used for cluster validation, confidence scoring, inspection, and reporting.

This approach provides a clear geometric explanation for why each document belongs to its cluster.

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

CLUSTER_TABLE = "noc_documents_clustering"      
SOURCE_TABLE  = "noc_documents_azure"           
OUT_EXPL_TABLE = "noc_documents_cluster_explainability"

TOP_REPS_PER_CLUSTER = 10
TOP_EDGE_PER_CLUSTER = 10

sdf = (
    spark.table(CLUSTER_TABLE)
         .select("path", "cluster", "cluster_keywords", "was_noise")
         .join(
             spark.table(SOURCE_TABLE).select("path", "final_text", "embedding"),
             on="path",
             how="inner"
         )
         .filter(F.col("embedding").isNotNull())
)

pdf = sdf.toPandas()

print(f"Loaded {len(pdf):,} clustered documents")

X = np.vstack(pdf["embedding"].values).astype(np.float32)

zero_mask = (X == 0).all(axis=1)
if zero_mask.any():
    pdf = pdf[~zero_mask].reset_index(drop=True)
    X = X[~zero_mask]

X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
labels = pdf["cluster"].astype(int).values

centroids = {}
for cid in sorted(pd.unique(labels)):
    idx = np.where(labels == cid)[0]
    c = X_norm[idx].mean(axis=0)
    c = c / (np.linalg.norm(c) + 1e-12)
    centroids[int(cid)] = c.astype(np.float32)

centroid_ids = np.array(list(centroids.keys()), dtype=np.int32)
C = np.vstack([centroids[c] for c in centroid_ids]).astype(np.float32)

print(f"Computed {len(centroid_ids)} centroids")

sims_all = X_norm @ C.T

cid_to_col = {int(cid): j for j, cid in enumerate(centroid_ids.tolist())}
own_cols = np.array([cid_to_col[int(c)] for c in labels], dtype=np.int32)

own_sim = sims_all[np.arange(len(pdf)), own_cols]


tmp = sims_all.copy()
tmp[np.arange(len(pdf)), own_cols] = -1e9

best_other_col = np.argmax(tmp, axis=1)
best_other_sim = tmp[np.arange(len(pdf)), best_other_col]
best_other_cid = centroid_ids[best_other_col]

margin = own_sim - best_other_sim

pdf["similarity_to_centroid"] = own_sim
pdf["closest_other_cluster"] = best_other_cid
pdf["similarity_to_other_centroid"] = best_other_sim
pdf["centroid_margin"] = margin
pdf["preview"] = pdf["final_text"].astype(str).str[:200].str.replace("\n"," ")


reps = (
    pdf.sort_values(["cluster","similarity_to_centroid"], ascending=[True,False])
       .groupby("cluster")
       .head(TOP_REPS_PER_CLUSTER)
       .copy()
)
reps["explain_role"] = "representative_core"

edges_margin = (
    pdf.sort_values(["cluster","centroid_margin"], ascending=[True,True])
       .groupby("cluster")
       .head(TOP_EDGE_PER_CLUSTER)
       .copy()
)
edges_margin["explain_role"] = "boundary_low_margin"

print("\nTop representative documents per cluster:")
display(
    spark.createDataFrame(
        reps[["cluster","similarity_to_centroid","centroid_margin","preview"]]
    )
)

print("\nBoundary documents (lowest margin):")
display(
    spark.createDataFrame(
        edges_margin[["cluster","similarity_to_centroid","centroid_margin","closest_other_cluster","preview"]]
    )
)


out_cols = [
    "path","cluster","cluster_keywords","was_noise",
    "similarity_to_centroid",
    "closest_other_cluster",
    "similarity_to_other_centroid",
    "centroid_margin"
]

spark.createDataFrame(pdf[out_cols]) \
     .write.mode("overwrite") \
     .option("mergeSchema","true") \
     .saveAsTable(OUT_EXPL_TABLE)

print(f"\nSaved explainability table → {OUT_EXPL_TABLE}")

In [0]:
from pyspark.sql import functions as F

CLUSTER_TABLE = "noc_documents_clustering"
SOURCE_TABLE  = "noc_documents_azure"

docs = (
    spark.table(CLUSTER_TABLE)
    .join(
        spark.table(SOURCE_TABLE).select("path", "final_text"),
        on="path",
        how="inner"
    )
)

# Split OCR text into rows
rows = docs.select(
    "path",
    F.explode(F.split(F.col("final_text"), "\n")).alias("row")
)

# Remove empty rows
rows = rows.filter(F.length(F.trim(F.col("row"))) > 0)

# Clean quotes and split columns
cols = rows.withColumn(
    "cols",
    F.split(F.regexp_replace("row", '"', ""), ",")
)

# Safe extraction using get()
result = cols.select(
    "path",
    F.expr("get(cols,0)").alias("NOC_NUMBER"),
    F.expr("get(cols,1)").alias("NOC_DATE"),
    F.expr("get(cols,2)").alias("NOC_MANUFACTURER_NAME"),
    F.expr("get(cols,3)").alias("NOC_STATUS_WITH_CONDITIONS"),
    F.expr("get(cols,4)").alias("NOC_ENG_SUBMISSION_TYPE"),
    F.expr("get(cols,5)").alias("NOC_FR_SUBMISSION_TYPE"),
    F.expr("get(cols,6)").alias("NOC_IS_SUPPLEMENT"),
    F.expr("get(cols,7)").alias("NOC_ENG_SUBMISSION_CLASS"),
    F.expr("get(cols,8)").alias("NOC_FR_SUBMISSION_CLASS"),
    F.expr("get(cols,9)").alias("NOC_ENG_PRODUCT_TYPE"),
    F.expr("get(cols,10)").alias("NOC_FR_PRODUCT_TYPE"),
    F.expr("get(cols,11)").alias("NOC_CRP_PRODUCT_NAME"),
    F.expr("get(cols,12)").alias("NOC_CRP_COMPANY_NAME"),
    F.expr("get(cols,13)").alias("NOC_CRP_COUNTRY_ENG_NAME"),
    F.expr("get(cols,14)").alias("NOC_CRP_COUNTRY_FR_NAME"),
    F.expr("get(cols,15)").alias("NOC_ACTIVE_STATUS"),
    F.expr("get(cols,16)").alias("NOC_ENG_REASON_SUPPLEMENT"),
    F.expr("get(cols,17)").alias("NOC_FR_REASON_SUPPLEMENT"),
    F.expr("get(cols,18)").alias("NOC_ENG_THERAPEUTIC_CLASS"),
    F.expr("get(cols,19)").alias("NOC_FR_THERAPEUTIC_CLASS"),
    F.expr("get(cols,20)").alias("NOC_ENG_REASON_SUBMISSION"),
    F.expr("get(cols,21)").alias("NOC_FR_REASON_SUBMISSION")
)

display(result.limit(50))

## Prototype / Exemplar Documents per Cluster

This block produces **human-friendly explainability** by selecting a small set of **prototype (exemplar) documents** for every cluster. These exemplars are the documents that best represent what the cluster is “about”.

First, we load the cluster assignments from `noc_documents_clustering` and join them with the source table `noc_documents_azure` to retrieve each document’s **embedding** and **final_text**. This gives us everything needed to compute similarity and display readable examples.

Next, we normalize all embeddings so cosine similarity can be computed efficiently using vector operations. We then compute a **centroid** for each cluster by averaging the embeddings of documents in that cluster and normalizing the centroid vector.

After that, we compute `similarity_to_centroid` for every document, which measures how close the document is to its cluster centroid. Higher similarity means the document is more “central” and more representative of the cluster.

Finally, for each cluster, we select the **top K documents** with the highest similarity to the centroid. These are saved as ranked exemplars in `noc_documents_cluster_exemplars` and displayed for review, making it easy to understand clusters through real example documents.

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

CLUSTER_TABLE   = "noc_documents_clustering"   
SOURCE_TABLE    = "noc_documents_azure"         
OUT_EXEMPLARS   = "noc_documents_cluster_exemplars"
TOP_K           = 5                            
PREVIEW_CHARS   = 300                          

sdf = (
    spark.table(CLUSTER_TABLE)
         .select("path", "cluster", "cluster_keywords", "was_noise")
         .join(
             spark.table(SOURCE_TABLE).select("path", "final_text", "embedding"),
             on="path",
             how="inner"
         )
         .filter(F.col("embedding").isNotNull())
)

pdf = sdf.toPandas()
print(f"Loaded {len(pdf):,} docs")

X = np.vstack(pdf["embedding"].values).astype(np.float32)

zero_mask = (X == 0).all(axis=1)
if zero_mask.any():
    print(f"Removing {int(zero_mask.sum())} zero-embeddings")
    pdf = pdf[~zero_mask].reset_index(drop=True)
    X = X[~zero_mask]

X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

labels = pdf["cluster"].astype(int).values

centroids = {}
for cid in sorted(pd.unique(labels)):
    idx = np.where(labels == cid)[0]
    c = X_norm[idx].mean(axis=0)
    c = c / (np.linalg.norm(c) + 1e-12)
    centroids[int(cid)] = c.astype(np.float32)

C_for_docs = np.vstack([centroids[int(c)] for c in labels]).astype(np.float32)  # (N, D)

sim_to_centroid = np.sum(X_norm * C_for_docs, axis=1)

pdf["similarity_to_centroid"] = sim_to_centroid.astype(float)

pdf["preview"] = (
    pdf["final_text"]
      .astype(str)
      .str.replace("\n", " ")
      .str.replace("\r", " ")
      .str[:PREVIEW_CHARS]
)

exemplars_pdf = (
    pdf.sort_values(["cluster", "similarity_to_centroid"], ascending=[True, False])
       .groupby("cluster", as_index=False)
       .head(TOP_K)
       .copy()
)

exemplars_pdf["exemplar_rank"] = (
    exemplars_pdf.groupby("cluster")["similarity_to_centroid"]
                 .rank(method="first", ascending=False)
                 .astype(int)
)

exemplars_pdf = exemplars_pdf[[
    "cluster",
    "exemplar_rank",
    "similarity_to_centroid",
    "cluster_keywords",
    "was_noise",
    "path",
    "preview"
]]

spark.createDataFrame(exemplars_pdf) \
     .write.mode("overwrite") \
     .option("mergeSchema", "true") \
     .saveAsTable(OUT_EXEMPLARS)

print(f"Saved exemplars table → {OUT_EXEMPLARS}")

display(
    spark.table(OUT_EXEMPLARS)
         .orderBy("cluster", "exemplar_rank")
)

## Topic-Based Cluster Explainability (c-TF-IDF + optional KeyBERT/YAKE/LLM)

This block explains each cluster using **topic keywords** extracted from the document text. It provides an interpretable, text-grounded summary of what each cluster is about.

First, we join the cluster assignments with `final_text` so every document has both a **cluster ID** and its **text content**. We then clean and normalize the text for consistent keyword extraction.

Next, we build one “cluster document” per cluster by concatenating text from up to a capped number of documents. This keeps the process scalable while still capturing the dominant language of each cluster.

We compute **c-TF-IDF (class-based TF-IDF)** using `CountVectorizer` and a cluster-level IDF formulation. This highlights terms that are frequent within a cluster but uncommon across other clusters, producing more distinctive keywords than standard TF-IDF.

For each cluster, we select the **top N c-TF-IDF terms** and save them into `noc_documents_cluster_topics_ctfidf` along with cluster size. These keywords act as a concise explanation of the cluster’s theme.

Optionally, the same cluster texts can be passed through **KeyBERT** (embedding-based keywords), **YAKE** (statistical keyword extraction), or an **LLM-based labeling** step that converts keywords and exemplar snippets into human-readable cluster labels.

In [0]:
import re
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

from sklearn.feature_extraction.text import CountVectorizer

spark = SparkSession.builder.getOrCreate()

CLUSTER_TABLE = "noc_documents_clustering"
SOURCE_TABLE  = "noc_documents_azure"

OUT_CTFIDF_TABLE     = "noc_documents_cluster_topics_ctfidf"
OUT_KEYBERT_TABLE    = "noc_documents_cluster_topics_keybert"
OUT_YAKE_TABLE       = "noc_documents_cluster_topics_yake"
OUT_LLM_LABELS_TABLE = "noc_documents_cluster_topics_llm"

TOP_N_WORDS = 12
MIN_DOCS_PER_CLUSTER = 5

MAX_DOCS_PER_CLUSTER_FOR_TEXT = 300   
MAX_CHARS_PER_CLUSTER_TEXT    = 300_000  

ENABLE_KEYBERT = False
ENABLE_YAKE    = False
ENABLE_LLM     = False

sdf = (
    spark.table(CLUSTER_TABLE).select("path", "cluster")
    .join(spark.table(SOURCE_TABLE).select("path", "final_text"), on="path", how="inner")
    .filter(F.col("final_text").isNotNull())
)

pdf = sdf.toPandas()
pdf["cluster"] = pdf["cluster"].astype(int)
pdf["final_text"] = pdf["final_text"].astype(str)

print(f"Loaded {len(pdf):,} docs with text + cluster")

def basic_clean(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    return s

def build_cluster_texts(df: pd.DataFrame):
    """
    Returns:
      cluster_ids: list[int]
      cluster_texts: list[str]  (one doc per cluster: concatenated text)
      cluster_sizes: list[int]
    """
    rows = []
    for cid, g in df.groupby("cluster"):
        n = len(g)
        if n < MIN_DOCS_PER_CLUSTER:
            continue

        if n > MAX_DOCS_PER_CLUSTER_FOR_TEXT:
            g2 = g.sample(n=MAX_DOCS_PER_CLUSTER_FOR_TEXT, random_state=42)
        else:
            g2 = g

        text = " ".join([basic_clean(t) for t in g2["final_text"].tolist()])
        text = text[:MAX_CHARS_PER_CLUSTER_TEXT]
        rows.append((int(cid), text, int(n)))

    rows.sort(key=lambda x: x[0])
    cluster_ids   = [r[0] for r in rows]
    cluster_texts = [r[1] for r in rows]
    cluster_sizes = [r[2] for r in rows]
    return cluster_ids, cluster_texts, cluster_sizes

cluster_ids, cluster_texts, cluster_sizes = build_cluster_texts(pdf)
print(f"Clusters used for topic extraction: {len(cluster_ids)}")

vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=50_000,
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{3,}\b",
)

X_counts = vectorizer.fit_transform(cluster_texts) 
terms = np.array(vectorizer.get_feature_names_out())

counts = X_counts.toarray().astype(np.float64)
tf = counts / (counts.sum(axis=1, keepdims=True) + 1e-12)

df_w = (counts > 0).sum(axis=0)

sum_w = counts.sum(axis=0)

idf = np.log((sum_w + 1.0) / (df_w + 1.0)) + 1.0
ctfidf = tf * idf  # (C, V)

topic_rows = []
for i, cid in enumerate(cluster_ids):
    top_idx = np.argsort(ctfidf[i])[-TOP_N_WORDS:][::-1]
    keywords = ", ".join(terms[top_idx])
    topic_rows.append({
        "cluster": int(cid),
        "doc_count": int(cluster_sizes[i]),
        "ctfidf_keywords": keywords
    })

topics_ctfidf_pdf = pd.DataFrame(topic_rows).sort_values("doc_count", ascending=False)

spark.createDataFrame(topics_ctfidf_pdf) \
     .write.mode("overwrite") \
     .option("mergeSchema", "true") \
     .saveAsTable(OUT_CTFIDF_TABLE)

print(f"Saved c-TF-IDF topics → {OUT_CTFIDF_TABLE}")
display(spark.table(OUT_CTFIDF_TABLE).orderBy(F.desc("doc_count")))

if ENABLE_KEYBERT:
    from keybert import KeyBERT

    kw_model = KeyBERT()

    keybert_rows = []
    for cid, text, n in zip(cluster_ids, cluster_texts, cluster_sizes):

        text_short = text[:60_000]
        kws = kw_model.extract_keywords(
            text_short,
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            top_n=TOP_N_WORDS,
            use_mmr=True,
            diversity=0.5
        )
        keywords = ", ".join([k for k, _ in kws])
        keybert_rows.append({
            "cluster": int(cid),
            "doc_count": int(n),
            "keybert_keywords": keywords
        })

    keybert_pdf = pd.DataFrame(keybert_rows).sort_values("doc_count", ascending=False)
    spark.createDataFrame(keybert_pdf) \
         .write.mode("overwrite") \
         .option("mergeSchema", "true") \
         .saveAsTable(OUT_KEYBERT_TABLE)

    print(f"Saved KeyBERT topics → {OUT_KEYBERT_TABLE}")
    display(spark.table(OUT_KEYBERT_TABLE).orderBy(F.desc("doc_count")))

if ENABLE_YAKE:
    import yake

    kw_extractor = yake.KeywordExtractor(
        lan="en",          
        n=2,
        top=TOP_N_WORDS
    )

    yake_rows = []
    for cid, text, n in zip(cluster_ids, cluster_texts, cluster_sizes):
        text_short = text[:120_000]
        kws = kw_extractor.extract_keywords(text_short)

        kws_sorted = sorted(kws, key=lambda x: x[1])[:TOP_N_WORDS]
        keywords = ", ".join([k for k, _ in kws_sorted])
        yake_rows.append({
            "cluster": int(cid),
            "doc_count": int(n),
            "yake_keywords": keywords
        })

    yake_pdf = pd.DataFrame(yake_rows).sort_values("doc_count", ascending=False)
    spark.createDataFrame(yake_pdf) \
         .write.mode("overwrite") \
         .option("mergeSchema", "true") \
         .saveAsTable(OUT_YAKE_TABLE)

    print(f"Saved YAKE topics → {OUT_YAKE_TABLE}")
    display(spark.table(OUT_YAKE_TABLE).orderBy(F.desc("doc_count")))

if ENABLE_LLM:

    EXEMPLARS_TABLE = "noc_documents_cluster_exemplars"  
    ex = spark.table(EXEMPLARS_TABLE).select("cluster", "exemplar_rank", "preview").toPandas()
    ex["cluster"] = ex["cluster"].astype(int)

    kw = spark.table(OUT_CTFIDF_TABLE).select("cluster", "ctfidf_keywords", "doc_count").toPandas()
    kw["cluster"] = kw["cluster"].astype(int)

    merged = kw.merge(
        ex.groupby("cluster")["preview"].apply(lambda s: list(s)[:5]).reset_index(name="exemplar_previews"),
        on="cluster",
        how="left"
    )

    def call_llm_for_label(cluster_id: int, keywords: str, previews: list[str]) -> str:
        prompt = f"""
You are labeling document clusters for regulatory text.
Return a short label (3-8 words) and a one-sentence rationale.

Cluster: {cluster_id}
Keywords: {keywords}
Exemplar snippets:
- """ + "\n- ".join((previews or [])[:5]) + "\n\nFormat:\nLabel: ...\nRationale: ...\n"
     
        return f"Label: {keywords.split(',')[0].strip().title()}\nRationale: Top keyword suggests main theme."

    llm_rows = []
    for _, r in merged.iterrows():
        out = call_llm_for_label(int(r["cluster"]), str(r["ctfidf_keywords"]), r.get("exemplar_previews", []))
        llm_rows.append({
            "cluster": int(r["cluster"]),
            "doc_count": int(r["doc_count"]),
            "ctfidf_keywords": str(r["ctfidf_keywords"]),
            "llm_label_raw": out
        })

    llm_pdf = pd.DataFrame(llm_rows)
    spark.createDataFrame(llm_pdf) \
         .write.mode("overwrite") \
         .option("mergeSchema", "true") \
         .saveAsTable(OUT_LLM_LABELS_TABLE)

    print(f"Saved LLM labels → {OUT_LLM_LABELS_TABLE}")
    display(spark.table(OUT_LLM_LABELS_TABLE).orderBy(F.desc("doc_count")))

print("Done. You have c-TF-IDF topics now. Enable KeyBERT/YAKE/LLM by toggling flags.")

## HDBSCAN Native Explainability (Persistence + Membership Strength)

This block reruns the clustering pipeline and extracts **native HDBSCAN explainability signals**. These signals explain cluster quality using density-based evidence rather than text keywords or centroid geometry.

First, we load document **embeddings** from `noc_documents_azure`, normalize them, and reduce dimensionality using **PCA** followed by **UMAP**. This creates a compact representation suitable for density-based clustering.

Next, we run **HDBSCAN** on the UMAP embeddings. HDBSCAN assigns a cluster label to each document (or `-1` for noise) and internally computes additional confidence diagnostics when `prediction_data=True`.

We then extract three key explainability metrics per document. **Membership strength** (`probabilities_`) represents how confidently the document belongs to its assigned cluster. **Outlier score** indicates how much a document behaves like an anomaly relative to its cluster. **Cluster persistence** measures how stable a cluster is across the density hierarchy.

Finally, we save two outputs. The **document-level table** stores each document’s cluster label and confidence metrics. The **cluster-level table** aggregates these metrics per cluster (size, persistence, average membership, and average outlier score) so we can identify the most stable and coherent clusters.

In [0]:
%pip install -q umap-learn hdbscan scikit-learn

import numpy as np
import pandas as pd
import umap
import hdbscan

from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

spark = SparkSession.builder.getOrCreate()

SOURCE_TABLE = "noc_documents_azure"

PCA_DIMS       = 50
UMAP_DIMS      = 15
UMAP_NEIGHBORS = 30
UMAP_MIN_DIST  = 0.0

MIN_CLUSTER_SIZE = 50
MIN_SAMPLES      = 5

OUT_DOC_TABLE     = "noc_documents_cluster_doc_explainability_hdbscan"
OUT_CLUSTER_TABLE = "noc_documents_cluster_cluster_explainability_hdbscan"

sdf = (
    spark.table(SOURCE_TABLE)
         .select("path", "embedding")
         .filter(F.col("embedding").isNotNull())
)

pdf = sdf.toPandas()
X = np.vstack(pdf["embedding"].values).astype(np.float32)

print(f"Loaded embeddings: {X.shape}")

X = normalize(X)

pca = PCA(n_components=PCA_DIMS, random_state=42)
X_pca = pca.fit_transform(X)

reducer = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    n_components=UMAP_DIMS,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    random_state=42,
    low_memory=True
)

X_umap = reducer.fit_transform(X_pca)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method="eom",
    prediction_data=True
)

labels = clusterer.fit_predict(X_umap)

pdf["cluster"] = labels
pdf["membership_strength"] = clusterer.probabilities_
pdf["outlier_score"] = clusterer.outlier_scores_

print("Clusters found:", len(set(labels)) - (1 if -1 in labels else 0))

unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
persistence_vals = clusterer.cluster_persistence_

persist_map = {
    int(cid): float(p)
    for cid, p in zip(unique_clusters, persistence_vals)
}

pdf["cluster_persistence"] = pdf["cluster"].map(
    lambda c: persist_map.get(int(c), np.nan)
)

doc_expl_pdf = pdf[[
    "path",
    "cluster",
    "membership_strength",
    "outlier_score",
    "cluster_persistence"
]]

spark.createDataFrame(doc_expl_pdf) \
     .write.mode("overwrite") \
     .saveAsTable(OUT_DOC_TABLE)

print("Saved →", OUT_DOC_TABLE)

cluster_rows = []
for cid in unique_clusters:
    g = pdf[pdf["cluster"] == cid]
    cluster_rows.append({
        "cluster": int(cid),
        "doc_count": int(len(g)),
        "cluster_persistence": float(persist_map.get(cid)),
        "avg_membership_strength": float(g["membership_strength"].mean()),
        "median_membership_strength": float(g["membership_strength"].median()),
        "avg_outlier_score": float(g["outlier_score"].mean())
    })

cluster_expl_pdf = pd.DataFrame(cluster_rows)

spark.createDataFrame(cluster_expl_pdf) \
     .write.mode("overwrite") \
     .saveAsTable(OUT_CLUSTER_TABLE)

print("Saved →", OUT_CLUSTER_TABLE)

display(
    spark.table(OUT_CLUSTER_TABLE)
         .orderBy(F.desc("cluster_persistence"))
)

## Results: Multi-Approach Explainability on the Same Clusters

This notebook evaluated four different explainability approaches on the **same HDBSCAN clusters**: centroid-based similarity, prototype/exemplar documents, topic-based labeling (c-TF-IDF and optional keyword methods), and native HDBSCAN density metrics. The goal was not to change clustering structure, but to assess how consistently different explanation lenses describe the same clusters.

The centroid-based approach showed that most documents have high similarity to their assigned cluster centroids, indicating strong geometric cohesion in embedding space. The centroid margin analysis revealed clear separation for core documents, while also identifying boundary documents that lie near competing clusters. This confirms that clusters are well-formed in vector space, with distinguishable centers and interpretable edge cases.

The prototype/exemplar extraction reinforced this finding. The top-ranked documents per cluster were thematically coherent and aligned well with the dominant patterns of their clusters. These exemplars provided intuitive, human-readable validation that the clusters represent meaningful document groupings rather than arbitrary density partitions.

The topic-based explanation using c-TF-IDF produced distinctive, cluster-specific keywords that aligned closely with the exemplar documents. This demonstrates semantic consistency between embedding-based structure and textual content. Optional methods such as KeyBERT, YAKE, and LLM-based labeling further enhanced interpretability by generating concise topic summaries, improving readability for downstream reporting or regulatory review.

Finally, the native HDBSCAN explainability metrics (membership strength, outlier score, and cluster persistence) provided density-based validation. Most documents exhibited high membership strength, indicating confident assignment. Several clusters demonstrated strong persistence, confirming stability across hierarchical density levels. Together, these metrics show that clusters are not only geometrically coherent but also density-stable.

Overall, the four explainability approaches converge: geometric similarity, representative exemplars, semantic keyword extraction, and density-based confidence all support the validity of the clustering structure. This multi-perspective agreement provides strong evidence that the clusters are meaningful, stable, and interpretable.